In [ ]:
import dijido
from datetime import timedelta
from datetime import datetime
import pandas
import numpy

from plotly import graph_objects
from plotly import offline as plotly

import plotly.io as pio
pio.renderers.default = "notebook"


import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

import scipy
from scipy import signal

In [ ]:
# Start date is inclusive, and begins at midnight the day of the start date
START_DATE = "2026-03-01"

# End date is inclusive, and includes all time up to midnight of the following day
END_DATE = "2026-03-18"

In [ ]:
dijido.login()

In [ ]:
goals = dijido.get_goals()

goals_dict = {
    goal["_id"]: goal
    for goal in goals
}

child_goal_lookup_dict = {
    goal["_id"]: set([])
    for goal in goals
}

for goal in goals:

    for goal_id in goal["parent_goal_ids"]:
        child_goal_lookup_dict[goal_id].add(goal["_id"])

In [ ]:
start_date = datetime.strptime(START_DATE, "%Y-%m-%d")
date = start_date
end_date = datetime.strptime(END_DATE, "%Y-%m-%d")

goals_by_date = {}

while date <= end_date:
    
    date_string = date.strftime("%Y-%m-%d")
    print(date_string)
    
    date_goals = dijido.get_goal_times_by_date_range(date_string, date_string, split_overlapping=True)
    goals_by_date[date_string] = date_goals
    
    date = date + timedelta(days=1)

In [ ]:
def get_children_goal_ids(goal_id):

    return child_goal_lookup_dict[goal_id]

def get_children_goal_ids_in_date_range(goal_id):

    children_goal_ids = set([])
    
    for date in goals_by_date:
        for goal_name, goal in goals_by_date[date].items():
            if "parent_goal_ids" not in goal:
                continue
            if goal_id in goal["parent_goal_ids"]:
                children_goal_ids.add(goal["_id"])
                
    return children_goal_ids

In [ ]:
def plot_time_trace(goal_name):

    traces = []

    date = datetime.strptime(START_DATE, "%Y-%m-%d")
    end_date = datetime.strptime(END_DATE, "%Y-%m-%d")

    xs = []
    ys = []

    while date <= end_date:

        days = (date - start_date).days
        xs.append(days)

        y = None

        date_string = date.strftime("%Y-%m-%d")

        if goal_name in goals_by_date[date_string]:
            y = goals_by_date[date_string][goal_name]["duration"] / 60 / 60
        elif not y:
            y = 0

        ys.append(y)

        date = date + timedelta(days=1)

    trace = graph_objects.Scatter(
        x=xs,
        y=ys,
        name=goal_name
    )

    traces.append(trace)

    layout = {
        "xaxis": {
            "tickmode": "array",
            "tickvals": xs,
            "ticktext": [(start_date + timedelta(days=x)).strftime("%a %m/%d") for x in xs]
        },
        "yaxis": {
            "title": "Hours per day"
        },
        "plot_bgcolor": "rgba(255, 255, 255, 0)",
        "paper_bgcolor": "rgba(255, 255, 255, 0)",
        "title": goal_name
    }

    figure = graph_objects.Figure(data=traces, layout=layout)

    plotly.iplot(figure)

In [ ]:
window_size = 1

def plot_time_traces_hierarchical(parent_goal_id, level=1):
    
    if level > num_levels:
        return
    
    child_goal_ids = get_children_goal_ids(parent_goal_id)
    child_goal_ids_in_date_range = get_children_goal_ids_in_date_range(parent_goal_id)
    
    parent_goal_name = dijido.get_goal_by_id(parent_goal_id)["name"]
    
    if len(child_goal_ids_in_date_range) > 0:
    
        traces = []
        data = []
    
        for child_goal_id in child_goal_ids_in_date_range:
    
            date = datetime.strptime(START_DATE, "%Y-%m-%d")
            end_date = datetime.strptime(END_DATE, "%Y-%m-%d")
    
            xs = []
            ys = []
    
            child_goal = dijido.get_goal_by_id(child_goal_id)
    
            while date <= end_date:
    
                days = (date - start_date).days
                xs.append(days)
    
                y = None
    
                date_string = date.strftime("%Y-%m-%d")
    
                if child_goal_id in goals_by_date[date_string]:
                    y = goals_by_date[date_string][child_goal_id]["duration"] / 60 / 60
                elif not y:
                    y = 0
    
                ys.append(y)
    
                date = date + timedelta(days=1)
            
            data.append({
                "name": child_goal["name"],
                "time (h)": numpy.sum(ys),
                "mean (h/d)": numpy.mean(ys),
                "mean (h/w)": numpy.mean(ys)*7
            })
    
    #         ys = numpy.convolve(ys, numpy.ones(window_size)/window_size, mode='valid') * 7
            
            trace = graph_objects.Scatter(
                x=xs,
                y=ys,
                name=child_goal["name"],
        #         line={'shape': 'spline', 'smoothing': 1.3}
            )
    
            traces.append(trace)
    
        layout = {
            "xaxis": {
                "tickmode": "array",
                "tickvals": xs[::window_size],
                "ticktext": [(start_date + timedelta(days=x)).strftime("%a %m/%d") for x in xs][::window_size]
            },
            "yaxis": {
                "title": "Hours per day"
            },
            "plot_bgcolor": "rgba(255, 255, 255, 0)",
            "paper_bgcolor": "rgba(255, 255, 255, 0)",
            "title": parent_goal_name
        }
    
        figure = graph_objects.Figure(data=traces, layout=layout)
        # plotly.iplot(figure)
        # figure.write_html(f"{parent_goal_name}.html")
        
        for datum in data:
            datum["percent"] = datum["mean (h/w)"] / numpy.sum([x["mean (h/w)"] for x in data])
            datum["percent_of_non_os"] = datum["mean (h/w)"] / numpy.sum([x["mean (h/w)"] for x in data if x["name"] != "Maintain Self OS"])
        
        df = pandas.DataFrame.from_records(data)
        print(parent_goal_name)
        display(df.sort_values(by="mean (h/d)", ascending=False))
    else:
        # print(f"No time logged for children of {parent_goal_name}")
        pass
    
    for child_goal_id in child_goal_ids:
        plot_time_traces_hierarchical(child_goal_id, level+1)

In [ ]:
pandas.options.display.min_rows = 50

In [ ]:
num_levels = 4
top_level_goal_name = "Optimize objective function (lol how meta)"
# top_level_goal_name = "Support Frontier Tower"
# top_level_goal_name = "Start or shape a community living movement"
# top_level_goal_name = "Get ripped (lol)"
# top_level_goal_name = "Manage life logistics"
# top_level_goal_name = "ImYoo - Admin"
# top_level_goal_name = "ImYoo"
# top_level_goal_name = "Propagate the world I want to see"
# top_level_goal_name = "Have adventures (digital)"
# top_level_goal_name = "Recuperate"
# top_level_goal_name = "Maintain Self OS"
# top_level_goal_name = "Enter the fight"
# top_level_goal_name = "Be a good citizen"
# top_level_goal_name = "Do work for Matt Thomson Lab"
# top_level_goal_name = "Maximize lifespan"
# top_level_goal_name = "Propagate Tatyana"
top_level_goal_id = dijido.get_goals_by_name(top_level_goal_name)[0]["_id"]

plot_time_traces_hierarchical(top_level_goal_id)

In [ ]:

from scipy.stats import gaussian_kde
from scipy import interpolate

In [ ]:
# child_goal_ids = get_children_goal_ids(dijido.get_goals_by_name("Optimize objective function"))

goals_of_interest = ["ImYoo", "Bathroom", top_level_goal_name, "Get good at muay thai/kickboxing", "Be a good rock climber", "Maintain strenuous exercise routine", "Untracked", "Change the world", "Experience life", "Procure, prepare, and consume food", "Establish and maintain connections with people", "Sleep", "Reduce risk of disease"]
# goals_of_interest = ["Procure, prepare, and consume food"]
traces = []
data = []

for goal_name in goals_of_interest:
    
    date = datetime.strptime(START_DATE, "%Y-%m-%d")
    end_date = datetime.strptime(END_DATE, "%Y-%m-%d")

    xs = []
    ys = []

    while date <= end_date:

        days = (date - start_date).days
        xs.append(days)

        y = None

        date_string = date.strftime("%Y-%m-%d")

        if goal_name in goals_by_date[date_string]:
            y = goals_by_date[date_string][goal_name]["duration"] / 60 / 60
        else:
            
            try:
                goals = dijido.get_goals_by_name(goal_name)
                goal_id = goals[0]["_id"]
                
                if goal_id in goals_by_date[date_string]:
                    y = goals_by_date[date_string][goal_id]["duration"] / 60 / 60
                else:
                    y = 0
            except:
                y = 0

        ys.append(y)

        date = date + timedelta(days=1)

    ys = numpy.convolve(ys, numpy.ones(window_size)/window_size, mode='valid') * 7
    trace = graph_objects.Scatter(
        x=xs,
        y=ys,
        name=goal_name,
        line={
            "width": 3
        }
    )

    data.append({
        "name": goal_name,
        "mean (h/d)": numpy.mean(ys)/7,
        "mean (h/w)": numpy.mean(ys),
    })

    traces.append(trace)

layout = {
    "xaxis": {
        "tickmode": "array",
        "tickvals": xs[::window_size],
        "ticktext": [(start_date + timedelta(days=x)).strftime("%a %m/%d") for x in xs][::window_size]
    },
    "yaxis": {
        "title": "Hours per week"
    },
    "plot_bgcolor": "rgba(255, 255, 255, 0)",
    "paper_bgcolor": "rgba(255, 255, 255, 0)",
    "title": "Time analysis",
    "width": 1000,
    "height": 1000
}



figure = graph_objects.Figure(data=traces, layout=layout)

plotly.iplot(figure)

df = pandas.DataFrame.from_records(data)
display(df.sort_values(by="mean (h/d)", ascending=False))

In [ ]:
datetime.strptime(START_DATE, "%Y-%m-%d")

In [ ]:
def plot_goal_time_usage_by_interval(goals_of_interest, goals_by_date, start_date, end_date, interval="month"):

    if interval == "month":
        pass
    else:
        raise NotImplementedError("One day")
    
    traces = []
    data = []
    
    for goal_name in goals_of_interest:
        
        start_date = datetime.strptime(start_date, "%Y-%m-%d")
        previous_date = start_date
        date = start_date
        end_date = datetime.strptime(end_date, "%Y-%m-%d")

        current_month = date.month
        current_month_date = date
    
        xs = []
        ys = []
        tickdates = []
        cumulative_seconds = 0
    
        while date <= end_date:
    
            date_month = date.month

            # If we've ticked over a month, time to add data
            if (date.year, date.month) > (current_month_date.year, current_month_date.month):
                
                # Calculate the numbers for this month
                num_days = (previous_date - current_month_date).days + 1
                days = (current_month_date - start_date).days
                xs.append(days)
                tickdates.append(current_month_date)
                ys.append(cumulative_seconds / num_days / 60 / 60 * 7)
                
                data.append({
                    "month": current_month_date,
                    "hours_per_week": ys[-1],
                    "goal_name": goal_name
                })
                
                # Increment the month
                cumulative_seconds = 0
                current_month = date.month
                current_month_date = date
    
            date_string = date.strftime("%Y-%m-%d")
    
            if goal_name in goals_by_date[date_string]:
                cumulative_seconds += goals_by_date[date_string][goal_name]["duration"]
            else:
                
                try:
                    goals = dijido.get_goals_by_name(goal_name)
                    goal_id = goals[0]["_id"]
                    
                    if goal_id in goals_by_date[date_string]:
                        cumulative_seconds += goals_by_date[date_string][goal_id]["duration"]
                except:
                    pass

            previous_date = date
            date = date + timedelta(days=1)

        # Calculate the numbers for this month
        num_days = (previous_date - current_month_date).days + 1
        days = (current_month_date - start_date).days
        xs.append(days)
        tickdates.append(current_month_date)
        ys.append(cumulative_seconds / num_days / 60 / 60 * 7)
                
        data.append({
            "month": current_month_date,
            "hours_per_week": ys[-1],
            "goal_name": goal_name
        })
        
        trace = graph_objects.Scatter(
            x=xs,
            y=ys,
            name=goal_name,
            line={
                "width": 3
            }
        )
    
        traces.append(trace)
        
    layout = {
        "xaxis": {
            "tickmode": "array",
            "tickvals": [(date - start_date).days for date in tickdates],
            "ticktext": [x.strftime("%b %Y") for x in tickdates]
        },
        "yaxis": {
            "title": "Hours per week",
            "rangemode": "tozero"
        },
        "plot_bgcolor": "rgba(255, 255, 255, 0)",
        "paper_bgcolor": "rgba(255, 255, 255, 0)",
        "title": "Time analysis",
        "width": 1000
    }
    
    
    figure = graph_objects.Figure(data=traces, layout=layout)
    
    plotly.iplot(figure)

    return data

In [ ]:
data = plot_goal_time_usage_by_interval(["Procure, prepare, and consume food"], goals_by_date, START_DATE, END_DATE, interval="month")

In [ ]:
pandas.DataFrame.from_records(data).to_csv("food_time_usage.csv")

In [ ]:
plot_goal_time_usage_by_interval(["Be altruistic"], goals_by_date, START_DATE, END_DATE, interval="month")

In [ ]:
# Get all activities that are part of a specific goal name. Get